In [1]:
import math
import re
from collections import Counter, defaultdict,OrderedDict
from itertools import combinations
from pathlib import Path

import networkx as nx
from IPython.display import IFrame
from pyvis.network import Network

import langextract as lx

In [2]:
PAGES_DIR = Path("../src/data/processed")
OUTPUT_HTML = "pages.html"

MAX_PAGES_PER_WORD = 3   # a word only links pages if it appears in at most this many pages
NEIGHBORS_PER_PAGE = 3   # each page keeps only its strongest links

In [4]:
def read_page(path):
    lines = path.read_text(encoding="utf-8").splitlines()
    number = int(lines[0])
    if lines[-1] == "NOTES":
        lines = lines[:-1]
    # join the lines with spaces instead of newlines
    text = " ".join(lines[1:])
    # rejoin words broken by hyphenation: "valetudi- narian", "con- cern"
    text = re.sub(r"(\w)- (\w)", r"\1\2", text)
    return number, text


pages = dict(read_page(p) for p in PAGES_DIR.glob("page_*.txt"))
pages = dict(sorted(pages.items()))


In [18]:
print(pages.get(1))

I sit down alone at the appointed table and take up my pen to give all whom it may concern an exact account of what may happen. Call me nervous, call me fey, if you will; at least this little pen, this mottled black and silver Aquarius, with its nib specially tempered to my order in Amsterdam, is greedy. It has not had much work since it flew so nimbly for the dead old man. As I watch the sea, Casy Ferris passes with down-dropped eyes. Of course, to-day is the day. Her father reminds me of a valetudinarian walrus. But she has, I suppose, to have somebody. St. Lazarus-in-the-Chine is full, no doubt, already. I think she is rash ; but it is none of my business. Where about the graves of the martyrs the whaups are crying, my heart remembers how. Strange that he comes into my head so much to-day. I hope it’s over some flotsam fish that the birds are making whaupee. But all the nice gulls love a sailor. Ugh.


In [21]:
import textwrap
import langextract as lx
from dotenv import load_dotenv

load_dotenv("../.env")  # only needed for Gemini

prompt = textwrap.dedent("""\
    Extract narrative information from a literary puzzle.
    Extraction classes and their attributes:
    - character: names, aliases or descriptive references to a person (e.g. "Her father").
      Attributes: mention ("name" | "alias" | "description").
    - event: actions or happenings; the span is the action itself.
      Attributes: actor, object, participants, location (only when explicitly stated).
    - temporal_cue: dates, times and sequence indicators.
      Attributes: kind ("date" | "time" | "relative" | "sequence").
    - location: places mentioned in the narrative.
      Attributes: context.
    - object: meaningful physical objects.
      Attributes: kind.
    - literary_reference: quotations, allusions or wordplay referring to literary works,
      songs or other texts, explicit or not.
      Attributes: kind ("quotation" | "allusion" | "wordplay"), suspected_source.

    Rules:
    1. Extract exact spans from the original text.
    2. Preserve the order of appearance.
    3. Do not overlap extraction spans.
    4. Only fill attributes with information explicitly stated in the text.
       The single exception is suspected_source, which is a hypothesis.
    5. Do not assume two aliases refer to the same character.
    6. Do not infer ordering between different pages.
    7. Extract evidence, not conclusions.
    """)

# the extraction_text values must appear exactly as written in example_text
example_text = (
    "At least this little pen, this mottled black and silver Aquarius, "
    "with its nib specially tempered to my order in Amsterdam, is greedy. "
    "As I watch the sea, Casy Ferris passes with down-dropped eyes. "
    "Of course, to-day is the day. Her father reminds me of a valetudinarian walrus. "
    "St. Lazarus-in-the-Chine is full, no doubt, already. "
    "Where about the graves of the martyrs the whaups are crying, my heart remembers how. "
    "But all the nice gulls love a sailor."
)

examples = [
    lx.data.ExampleData(
        text=example_text,
        extractions=[
            lx.data.Extraction(extraction_class="object",
                               extraction_text="mottled black and silver Aquarius",
                               attributes={"kind": "pen"}),
            lx.data.Extraction(extraction_class="narrator_clue", extraction_text="tempered to my order",
                               attributes={"reveals": "had the pen's nib specially tempered"}),
            lx.data.Extraction(extraction_class="location", extraction_text="Amsterdam",
                               attributes={"context": "where the pen's nib was tempered"}),
            lx.data.Extraction(extraction_class="event", extraction_text="watch",
                               attributes={"actor": "narrator", "object": "the sea"}),
            lx.data.Extraction(extraction_class="location", extraction_text="the sea",
                               attributes={"context": "watched by the narrator"}),
            lx.data.Extraction(extraction_class="event", extraction_text="passes with down-dropped eyes",
                               attributes={"actor": "Casy Ferris"}),
            lx.data.Extraction(extraction_class="temporal_cue", extraction_text="to-day is the day",
                               attributes={"kind": "relative"}),
            lx.data.Extraction(extraction_class="character", extraction_text="Her father",
                               attributes={"mention": "description"}),
            lx.data.Extraction(extraction_class="location", extraction_text="St. Lazarus-in-the-Chine",
                               attributes={"context": "is full, no doubt, already"}),
            lx.data.Extraction(extraction_class="literary_reference",
                               extraction_text="Where about the graves of the martyrs the whaups are crying",
                               attributes={"kind": "quotation",
                                           "suspected_source": "Robert Louis Stevenson, 'To S. R. Crockett'"}),
            lx.data.Extraction(extraction_class="literary_reference",
                               extraction_text="all the nice gulls love a sailor",
                               attributes={"kind": "wordplay",
                                           "suspected_source": "song 'All the Nice Girls Love a Sailor'"}),
        ],
    )
]


In [22]:
result = lx.extract(
    text_or_documents=pages[2],
    prompt_description=prompt,
    examples=examples,
    model_id="qwen3.5:2b",
    model_url="http://localhost:11434",
    fence_output=False,
    use_schema_constraints=False,
)


LangExtract: Processing, current=1,006 chars, processed=0 chars:  [00:25]


In [23]:
lx.io.save_annotated_documents([result], output_name="extraction_example.jsonl", output_dir=".")
lx.visualize("extraction_example.jsonl")


LangExtract: Saving to extraction_example.jsonl: 1 docs [00:00, 280.50 docs/s]

✓ Saved 1 documents to extraction_example.jsonl



LangExtract: Loading extraction_example.jsonl: 100%|█████████▉| 6.51k/6.51k [00:00<00:00, 279kB/s]

✓ Loaded 1 documents from extraction_example.jsonl
